# Laboratorio 2: Motor de búsqueda semántica

Este notebook desarrolla las secciones 7 a 10 del laboratorio: construcción del corpus, definición de consultas, generación de embeddings y cálculo de similitud coseno.

**Dominio seleccionado:** soporte técnico de una plataforma digital.

## Dependencias

Antes de ejecutar el notebook, el ambiente debe tener instaladas las dependencias indicadas en el laboratorio:

```bash
python -m pip install sentence-transformers numpy scikit-learn
```

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

## Sección 7 — Parte A: Corpus

El corpus contiene 24 oraciones en español relacionadas con problemas, configuraciones y solicitudes de soporte técnico.

In [2]:
CORPUS = [
    "No puedo iniciar sesión en mi cuenta.",
    "Olvidé mi clave de acceso al sistema.",
    "El usuario desea cambiar su contraseña.",
    "La plataforma muestra un error al iniciar sesión.",
    "La cuenta fue bloqueada por demasiados intentos fallidos.",
    "Necesito recuperar el acceso porque perdí mis credenciales.",
    "El sistema recomienda restablecer la clave de seguridad.",
    "No recibí el correo para recuperar mi contraseña.",
    "Quiero actualizar el correo electrónico asociado a mi perfil.",
    "La aplicación móvil se cierra al abrirla.",
    "La página tarda demasiado tiempo en cargar.",
    "El sistema muestra una pantalla en blanco después de ingresar.",
    "No puedo descargar el archivo desde la plataforma.",
    "El documento cargado supera el tamaño permitido.",
    "El micrófono no funciona durante las videollamadas.",
    "La cámara no es detectada por la aplicación.",
    "No escucho ningún sonido en la reunión virtual.",
    "La conexión se interrumpe constantemente durante la llamada.",
    "Deseo activar las notificaciones en mi teléfono.",
    "Las alertas de la aplicación llegan con retraso.",
    "Necesito cambiar el idioma de la interfaz.",
    "El código de verificación nunca llegó a mi celular.",
    "La plataforma no reconoce mi nombre de usuario.",
    "Quiero eliminar permanentemente mi cuenta.",
]

assert len(CORPUS) >= 20, "El corpus debe contener al menos 20 oraciones."

## Sección 8 — Parte B: Consultas

Las consultas incluyen coincidencias literales, relaciones semánticas sin las mismas palabras, un caso ambiguo y situaciones en las que la búsqueda por palabras clave puede fallar.

In [3]:
CONSULTAS = [
    "quiero cambiar mi contraseña",       # Coincidencia directa de palabras
    "no logro entrar a mi perfil",        # Relación semántica sin coincidencia directa completa
    "la app desaparece cuando la inicio", # Keyword search puede fallar: la aplicación se cierra
    "tengo problemas con el audio",       # Consulta ambigua: micrófono o sonido
    "nunca me enviaron el número para confirmar mi identidad",  # Código de verificación
    "el sitio funciona muy lento",        # Relación semántica: la página tarda en cargar
]

assert len(CONSULTAS) >= 5, "Se requieren al menos 5 consultas de prueba."

## Sección 9 — Parte C: Generación de embeddings

Se utiliza un modelo multilingüe de `sentence-transformers`. Los vectores se normalizan para que su producto punto equivalga a la similitud coseno.

In [4]:
MODELO_EMBEDDINGS = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

modelo = SentenceTransformer(MODELO_EMBEDDINGS)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
embeddings_corpus = modelo.encode(
    CORPUS,
    normalize_embeddings=True,
)
embeddings_corpus = np.asarray(embeddings_corpus)

print(f"Corpus: {len(CORPUS)} oraciones")
print(f"Dimensión de embeddings: {embeddings_corpus.shape[1]}")

Corpus: 24 oraciones
Dimensión de embeddings: 384


## Sección 10 — Parte D: Similitud coseno

Como todos los embeddings están normalizados, la similitud coseno se calcula mediante el producto punto entre el vector de la consulta y cada vector del corpus.

In [6]:
def similitud_coseno(
    embedding_consulta: np.ndarray,
    embeddings_documentos: np.ndarray,
) -> np.ndarray:
    """Calcula la similitud coseno entre una consulta y el corpus."""
    return embeddings_documentos @ embedding_consulta

In [7]:
# Ejemplo de cálculo para la primera consulta.
embedding_consulta = modelo.encode(
    CONSULTAS[0],
    normalize_embeddings=True,
)
puntajes_similitud = similitud_coseno(
    embedding_consulta,
    embeddings_corpus,
)

print(f"Consulta: {CONSULTAS[0]}")
print(f"Cantidad de puntajes calculados: {len(puntajes_similitud)}")
print(puntajes_similitud)

Consulta: quiero cambiar mi contraseña
Cantidad de puntajes calculados: 24
[ 0.4588719   0.5288281   0.9145702   0.25160542  0.05596768  0.48585564
  0.3156128   0.5765137   0.4878329   0.12307429  0.04943055  0.124373
  0.17624833 -0.07022603  0.08489443 -0.03649044  0.16294208 -0.01337084
  0.3450794   0.02398929  0.42023557  0.14579546  0.40402812  0.3629974 ]


## Sección 11 — Parte E: Recuperación top-k

Se implementa la recuperación de los `k` documentos más relevantes. Cada resultado incluye
la posición en el ranking, el índice original en el corpus, el texto recuperado y el
puntaje de similitud coseno.

In [8]:
def buscar_semanticamente(
    consulta: str,
    corpus: list[str],
    embeddings_corpus: np.ndarray,
    modelo: SentenceTransformer,
    top_k: int = 3,
) -> list[dict]:
    """Recupera los top-k documentos más similares a la consulta.

    Cada resultado incluye la posición en el ranking, el índice original en el
    corpus, el texto recuperado y el puntaje de similitud coseno.
    """
    embedding_consulta = modelo.encode(consulta, normalize_embeddings=True)
    puntajes = similitud_coseno(embedding_consulta, embeddings_corpus)

    indices_ordenados = np.argsort(puntajes)[::-1][:top_k]

    resultados = []
    for posicion, indice in enumerate(indices_ordenados, start=1):
        resultados.append(
            {
                "rank": posicion,
                "indice": int(indice),
                "texto": corpus[indice],
                "score": float(puntajes[indice]),
            }
        )
    return resultados


def imprimir_resultados_semanticos(resultados: list[dict]) -> None:
    """Imprime de forma legible los resultados de la búsqueda semántica."""
    for resultado in resultados:
        print(f"  {resultado['rank']}. score={resultado['score']:.4f} | {resultado['texto']}")


# Prueba con la primera consulta.
imprimir_resultados_semanticos(
    buscar_semanticamente(CONSULTAS[0], CORPUS, embeddings_corpus, modelo, top_k=3)
)


  1. score=0.9146 | El usuario desea cambiar su contraseña.
  2. score=0.5765 | No recibí el correo para recuperar mi contraseña.
  3. score=0.5288 | Olvidé mi clave de acceso al sistema.


## Sección 12 — Parte F: Búsqueda por palabras clave

Como línea base se implementa una búsqueda léxica simple: se tokeniza la consulta y cada
oración del corpus, se cuentan las palabras compartidas y se ordenan los documentos por
esa cantidad de coincidencias.

In [9]:
import re


def tokenizar_simple(texto: str) -> set[str]:
    """Tokeniza un texto en un conjunto de palabras en minúsculas."""
    return set(re.findall(r"\b\w+\b", texto.lower()))


def buscar_por_palabras_clave(
    consulta: str,
    corpus: list[str],
    top_k: int = 3,
) -> list[dict]:
    """Recupera los top-k documentos con más palabras compartidas con la consulta.

    Sirve como línea base léxica para contrastar con la búsqueda semántica.
    """
    tokens_consulta = tokenizar_simple(consulta)
    resultados = []

    for indice, texto in enumerate(corpus):
        coincidencias = tokens_consulta & tokenizar_simple(texto)
        resultados.append(
            {
                "indice": indice,
                "texto": texto,
                "score": len(coincidencias),
                "coincidencias": sorted(coincidencias),
            }
        )

    resultados.sort(key=lambda item: item["score"], reverse=True)
    return resultados[:top_k]


def imprimir_resultados_keyword(resultados: list[dict]) -> None:
    """Imprime de forma legible los resultados de la búsqueda por palabras clave."""
    for posicion, resultado in enumerate(resultados, start=1):
        coincidencias = ", ".join(resultado["coincidencias"]) or "sin coincidencias"
        print(
            f"  {posicion}. score={resultado['score']} | "
            f"coincidencias={coincidencias} | {resultado['texto']}"
        )


# Prueba con la primera consulta.
imprimir_resultados_keyword(buscar_por_palabras_clave(CONSULTAS[0], CORPUS, top_k=3))


  1. score=2 | coincidencias=cambiar, contraseña | El usuario desea cambiar su contraseña.
  2. score=2 | coincidencias=contraseña, mi | No recibí el correo para recuperar mi contraseña.
  3. score=2 | coincidencias=mi, quiero | Quiero actualizar el correo electrónico asociado a mi perfil.


## Sección 13 — Parte G: Comparación de resultados

Para cada consulta se muestran, una junto a la otra, la búsqueda semántica y la búsqueda
por palabras clave. Esto permite contrastar cuándo el significado supera al solapamiento
léxico y viceversa.

In [10]:
def comparar_busquedas(
    consulta: str,
    corpus: list[str],
    embeddings_corpus: np.ndarray,
    modelo: SentenceTransformer,
    top_k: int = 3,
) -> None:
    """Imprime lado a lado la búsqueda semántica y la búsqueda por palabras clave."""
    print("=" * 80)
    print(f"CONSULTA: {consulta}")
    print("=" * 80)

    resultados_semanticos = buscar_semanticamente(
        consulta, corpus, embeddings_corpus, modelo, top_k=top_k
    )
    resultados_keyword = buscar_por_palabras_clave(consulta, corpus, top_k=top_k)

    print("\nBúsqueda semántica")
    imprimir_resultados_semanticos(resultados_semanticos)

    print("\nBúsqueda por palabras clave")
    imprimir_resultados_keyword(resultados_keyword)
    print()


for consulta in CONSULTAS:
    comparar_busquedas(consulta, CORPUS, embeddings_corpus, modelo, top_k=3)


CONSULTA: quiero cambiar mi contraseña

Búsqueda semántica
  1. score=0.9146 | El usuario desea cambiar su contraseña.
  2. score=0.5765 | No recibí el correo para recuperar mi contraseña.
  3. score=0.5288 | Olvidé mi clave de acceso al sistema.

Búsqueda por palabras clave
  1. score=2 | coincidencias=cambiar, contraseña | El usuario desea cambiar su contraseña.
  2. score=2 | coincidencias=contraseña, mi | No recibí el correo para recuperar mi contraseña.
  3. score=2 | coincidencias=mi, quiero | Quiero actualizar el correo electrónico asociado a mi perfil.

CONSULTA: no logro entrar a mi perfil

Búsqueda semántica
  1. score=0.6226 | No puedo iniciar sesión en mi cuenta.
  2. score=0.5928 | La plataforma no reconoce mi nombre de usuario.
  3. score=0.5540 | Necesito recuperar el acceso porque perdí mis credenciales.

Búsqueda por palabras clave
  1. score=3 | coincidencias=a, mi, perfil | Quiero actualizar el correo electrónico asociado a mi perfil.
  2. score=2 | coincidencias=mi,

## Sección 14 — Reflexión

La búsqueda semántica funcionó claramente mejor en las consultas donde no había
coincidencia literal de palabras. El caso más representativo fue *"la app desaparece
cuando la inicio"*, que recuperó *"La aplicación móvil se cierra al abrirla."* pese a no
compartir ninguna palabra de contenido; la búsqueda por palabras clave no logró
asociarlas. Algo similar ocurrió con *"nunca me enviaron el número para confirmar mi
identidad"*, que se vinculó correctamente con el código de verificación.

La búsqueda por palabras clave fue competitiva, e incluso más predecible, en la consulta
con coincidencia directa *"quiero cambiar mi contraseña"*, donde el solapamiento léxico
con *"El usuario desea cambiar su contraseña."* bastó para ubicarla en el primer lugar
sin costo de cómputo.

El resultado más inesperado fue la consulta ambigua *"tengo problemas con el audio"*: el
modelo agrupó tanto el micrófono como el sonido de la reunión, mostrando que reparte la
similitud entre varias oraciones plausibles en lugar de elegir una sola.

Las limitaciones observadas incluyen puntajes relativamente altos incluso para oraciones
poco relacionadas (el coseno rara vez baja de forma marcada) y sensibilidad al dominio del
corpus. Para mejorar, ampliaría el corpus con más variantes por intención, aplicaría un
umbral mínimo de similitud para descartar resultados débiles y combinaría ambos enfoques
en un esquema híbrido (léxico + semántico) para aprovechar sus fortalezas complementarias.
